In [29]:
import pandas as pd
from pandas.io.sas.sas_constants import column_name_pointer_length
from sqlalchemy import create_engine

In [4]:
df = pd.read_csv('customer_shopping_behavior.csv')

In [31]:
df.head(10)

,customer_id,age,gender,item_purchased,category,Purchase Amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle Age,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle Age,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle Age,265
5,6,46,Male,Sneakers,Footwear,20,Wyoming,M,White,Summer,2.9,Yes,Standard,Yes,14,Venmo,Weekly,Middle Age,7
6,7,63,Male,Shirt,Clothing,85,Montana,M,Gray,Fall,3.2,Yes,Free Shipping,Yes,49,Cash,Quarterly,Senior,90
7,8,27,Male,Shorts,Clothing,34,Louisiana,L,Charcoal,Winter,3.2,Yes,Free Shipping,Yes,19,Credit Card,Weekly,Young Adult,7
8,9,26,Male,Coat,Outerwear,97,West Virginia,L,Silver,Summer,2.6,Yes,Express,Yes,8,Venmo,Annually,Young Adult,265
9,10,57,Male,Handbag,Accessories,31,Missouri,M,Pink,Spring,4.8,Yes,2-Day Shipping,Yes,4,Cash,Quarterly,Middle Age,90


In [ ]:
df.describe(include='all')

In [ ]:
df.info()

In [ ]:
df.isna().sum()

In [12]:
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(lambda x: x.fillna(x.median()))

In [13]:
df.isna().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [16]:
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.rename(columns={"purchase_amount_(usd)" : "Purchase Amount"})

In [21]:
labels = ["Young Adult", "Adult", "Middle Age", "Senior"]
df['age_group'] = pd.qcut(df['age'], q= 4, labels=labels )
df[['age','age_group']]

,age,age_group
0,55,Middle Age
1,19,Young Adult
2,50,Middle Age
3,21,Young Adult
4,45,Middle Age
...,...,...
3895,40,Adult
3896,52,Middle Age
3897,46,Middle Age
3898,44,Adult


In [24]:
frequency_mapping = {"Fortnightly" : 14, "Weekly": 7, 'Monthly': 30, 'Quarterly': 90, 'Bi-Weekly': 14, 'Annually': 265, 'Every 3 Months' : 90 }
df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)
df[['purchase_frequency_days','frequency_of_purchases']]

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,265,Annually
...,...,...
3895,7,Weekly
3896,14,Bi-Weekly
3897,90,Quarterly
3898,7,Weekly


In [25]:
(df['discount_applied'] == df['promo_code_used']).all()

np.True_

In [26]:
df =df.drop('promo_code_used', axis=1)

In [30]:
username = 'postgres'
password = 'ROOTanbu_007Kakashi'
host = 'localhost'
port = '5432'
database = 'customer_behavior'

engine = create_engine(
    f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}'
)

table_name = 'customer'
df.to_sql(
    table_name,
    engine, if_exists='replace',index=False,
)

print(f'Successfully created table {table_name} to database {database}')

Successfully created table customer to database customer_behavior
